# Matrix Decompositions - Exercises

Practice problems for LU, QR, and Cholesky decompositions.

## Contents
1. Manual LU Decomposition
2. Solve System with LU
3. QR via Gram-Schmidt
4. Least Squares via QR
5. Manual Cholesky
6. Gaussian Sampling
7. Determinant from Cholesky
8. LDLᵀ Decomposition
9. Condition Number Impact
10. Ridge Regression via Cholesky

In [ ]:
import numpy as np
from numpy.linalg import qr, cholesky, solve, det, inv, eigvalsh
from scipy.linalg import lu, lu_factor, lu_solve
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

---
## Exercise 1: Manual LU Decomposition

Find the LU decomposition (without pivoting) of:

$$A = \begin{pmatrix} 2 & 1 \\ 6 & 4 \end{pmatrix}$$

Verify that $A = L \cdot U$.

In [ ]:
A = np.array([[2, 1],
              [6, 4]], dtype=float)

# Hint: To eliminate A[1,0], use multiplier m = A[1,0]/A[0,0]
# L has 1s on diagonal, multiplier below diagonal
# U has the eliminated matrix entries

# YOUR CODE HERE

In [ ]:
# SOLUTION
A = np.array([[2, 1],
              [6, 4]], dtype=float)

print(f"A = \n{A}")

print("\n--- Gaussian Elimination ---")
print("To eliminate A[1,0] = 6:")
print("m = A[1,0]/A[0,0] = 6/2 = 3")
print("Row2 = Row2 - 3*Row1 = [6,4] - 3*[2,1] = [0,1]")

L = np.array([[1, 0],
              [3, 1]], dtype=float)  # Multiplier is 6/2 = 3

U = np.array([[2, 1],
              [0, 1]], dtype=float)  # [6,4] - 3*[2,1] = [0,1]

print(f"\nL = \n{L}")
print(f"U = \n{U}")

print(f"\nVerification: L @ U = \n{L @ U}")
print(f"Equals A? {np.allclose(L @ U, A)} ✓")

---
## Exercise 2: Solve System with LU

Use LU decomposition to solve:

$$\begin{pmatrix} 3 & 1 \\ 6 & 4 \end{pmatrix} \begin{pmatrix} x \\ y \end{pmatrix} = \begin{pmatrix} 5 \\ 14 \end{pmatrix}$$

In [ ]:
A = np.array([[3, 1],
              [6, 4]], dtype=float)
b = np.array([5, 14], dtype=float)

# Step 1: Find LU decomposition
# Step 2: Forward solve Ly = b
# Step 3: Back solve Ux = y

# YOUR CODE HERE

In [ ]:
# SOLUTION
A = np.array([[3, 1],
              [6, 4]], dtype=float)
b = np.array([5, 14], dtype=float)

print(f"A = \n{A}")
print(f"b = {b}")

# Step 1: LU decomposition
print("\n--- Step 1: LU Decomposition ---")
# Multiplier: 6/3 = 2
L = np.array([[1, 0],
              [2, 1]], dtype=float)
# Row2 - 2*Row1: [6,4] - 2*[3,1] = [0,2]
U = np.array([[3, 1],
              [0, 2]], dtype=float)

print(f"L = \n{L}")
print(f"U = \n{U}")

# Step 2: Forward solve Ly = b
print("\n--- Step 2: Forward Solve Ly = b ---")
y = np.zeros(2)
y[0] = b[0]  # y[0] = 5
y[1] = b[1] - L[1,0]*y[0]  # y[1] = 14 - 2*5 = 4
print(f"y[0] = b[0] = {y[0]}")
print(f"y[1] = b[1] - L[1,0]*y[0] = 14 - 2*5 = {y[1]}")
print(f"y = {y}")

# Step 3: Back solve Ux = y
print("\n--- Step 3: Back Solve Ux = y ---")
x = np.zeros(2)
x[1] = y[1] / U[1,1]  # x[1] = 4/2 = 2
x[0] = (y[0] - U[0,1]*x[1]) / U[0,0]  # x[0] = (5 - 1*2)/3 = 1
print(f"x[1] = y[1]/U[1,1] = 4/2 = {x[1]}")
print(f"x[0] = (y[0] - U[0,1]*x[1])/U[0,0] = (5 - 1*2)/3 = {x[0]}")
print(f"x = {x}")

print(f"\nVerification: A @ x = {A @ x}")

---
## Exercise 3: QR via Gram-Schmidt

Find the QR decomposition of:

$$A = \begin{pmatrix} 1 & 1 \\ 1 & 0 \\ 0 & 1 \end{pmatrix}$$

In [ ]:
A = np.array([[1, 1],
              [1, 0],
              [0, 1]], dtype=float)

# Hint: 
# q_1 = a_1 / ||a_1||
# v_2 = a_2 - (q_1^T a_2) q_1
# q_2 = v_2 / ||v_2||

# YOUR CODE HERE

In [ ]:
# SOLUTION
A = np.array([[1, 1],
              [1, 0],
              [0, 1]], dtype=float)

print(f"A = \n{A}")

print("\n--- Gram-Schmidt ---")

# Column 1
a1 = A[:, 0]
print(f"a₁ = {a1}")
r11 = np.linalg.norm(a1)
q1 = a1 / r11
print(f"||a₁|| = r₁₁ = {r11:.4f}")
print(f"q₁ = a₁/r₁₁ = {np.round(q1, 4)}")

# Column 2
a2 = A[:, 1]
print(f"\na₂ = {a2}")
r12 = q1 @ a2
print(f"r₁₂ = q₁ᵀa₂ = {r12:.4f}")
v2 = a2 - r12 * q1
print(f"v₂ = a₂ - r₁₂q₁ = {np.round(v2, 4)}")
r22 = np.linalg.norm(v2)
q2 = v2 / r22
print(f"r₂₂ = ||v₂|| = {r22:.4f}")
print(f"q₂ = v₂/r₂₂ = {np.round(q2, 4)}")

Q = np.column_stack([q1, q2])
R = np.array([[r11, r12], [0, r22]])

print(f"\nQ = \n{np.round(Q, 4)}")
print(f"R = \n{np.round(R, 4)}")
print(f"\nQᵀQ = \n{np.round(Q.T @ Q, 4)} (should be I)")
print(f"\nQ @ R = \n{np.round(Q @ R, 4)}")

---
## Exercise 4: Least Squares via QR

Use QR to find the least squares solution to:

$$\begin{pmatrix} 1 \\ 1 \\ 1 \end{pmatrix} x \approx \begin{pmatrix} 1 \\ 2 \\ 3 \end{pmatrix}$$

In [ ]:
A = np.array([[1], [1], [1]], dtype=float)
b = np.array([1, 2, 3], dtype=float)

# Use QR: x = R^(-1) Q^T b

# YOUR CODE HERE

In [ ]:
# SOLUTION
A = np.array([[1], [1], [1]], dtype=float)
b = np.array([1, 2, 3], dtype=float)

print(f"A = {A.flatten()}")
print(f"b = {b}")

print("\n--- QR Decomposition ---")
Q, R = qr(A, mode='reduced')
print(f"Q = {np.round(Q.flatten(), 4)} (3×1)")
print(f"R = [[{R[0,0]:.4f}]] (1×1)")

print("\n--- Solve Rx = Qᵀb ---")
Qb = Q.T @ b
print(f"Qᵀb = {Qb}")
x = Qb / R[0, 0]
print(f"x = Qᵀb / R = {x}")

print(f"\nSolution: x = {x[0]:.4f}")
print("(This is the mean of b: (1+2+3)/3 = 2)")

print(f"\nResidual: ||Ax - b|| = {np.linalg.norm(A @ x - b):.4f}")

---
## Exercise 5: Manual Cholesky

Find the Cholesky decomposition of:

$$A = \begin{pmatrix} 4 & 6 \\ 6 & 13 \end{pmatrix}$$

In [ ]:
A = np.array([[4, 6],
              [6, 13]], dtype=float)

# Formulas:
# l_11 = sqrt(a_11)
# l_21 = a_21 / l_11
# l_22 = sqrt(a_22 - l_21^2)

# YOUR CODE HERE

In [ ]:
# SOLUTION
A = np.array([[4, 6],
              [6, 13]], dtype=float)

print(f"A = \n{A}")

print("\n--- Cholesky Formula ---")
print("L = [[l₁₁, 0  ],")
print("     [l₂₁, l₂₂]]")

l11 = np.sqrt(A[0,0])
l21 = A[1,0] / l11
l22 = np.sqrt(A[1,1] - l21**2)

print(f"\nl₁₁ = √a₁₁ = √4 = {l11}")
print(f"l₂₁ = a₂₁/l₁₁ = 6/2 = {l21}")
print(f"l₂₂ = √(a₂₂ - l₂₁²) = √(13 - 9) = √4 = {l22}")

L = np.array([[l11, 0],
              [l21, l22]], dtype=float)

print(f"\nL = \n{L}")
print(f"\nVerification: L @ Lᵀ = \n{L @ L.T}")

# Verify with numpy
L_np = cholesky(A)
print(f"\nNumPy Cholesky: \n{L_np}")

---
## Exercise 6: Gaussian Sampling

Use Cholesky to sample from $\mathcal{N}(\mu, \Sigma)$ where:

$$\mu = \begin{pmatrix} 0 \\ 0 \end{pmatrix}, \quad \Sigma = \begin{pmatrix} 1 & 0.5 \\ 0.5 & 1 \end{pmatrix}$$

In [ ]:
mu = np.array([0, 0])
Sigma = np.array([[1, 0.5],
                 [0.5, 1]])

# Algorithm:
# 1. Compute L = cholesky(Sigma)
# 2. Sample z ~ N(0, I)
# 3. Return x = mu + L @ z

# YOUR CODE HERE

In [ ]:
# SOLUTION
mu = np.array([0, 0])
Sigma = np.array([[1, 0.5],
                 [0.5, 1]])

print(f"μ = {mu}")
print(f"Σ = \n{Sigma}")

# Cholesky
L = cholesky(Sigma)
print(f"\nL = cholesky(Σ) = \n{np.round(L, 4)}")

# Sample
print("\n--- Sampling Algorithm ---")
print("1. Sample z ~ N(0, I)")
print("2. Compute x = μ + L @ z")

np.random.seed(0)
print("\nFirst 5 samples:")
for i in range(5):
    z = np.random.randn(2)
    x = mu + L @ z
    print(f"  z = {np.round(z, 3)} → x = {np.round(x, 3)}")

In [ ]:
# Visualize samples
np.random.seed(42)
samples = np.array([mu + L @ np.random.randn(2) for _ in range(300)])

plt.figure(figsize=(8, 6))
plt.scatter(samples[:, 0], samples[:, 1], alpha=0.4, s=20)
plt.scatter([0], [0], c='red', s=100, marker='*', zorder=5, label='Mean')
plt.xlabel('$x_1$'); plt.ylabel('$x_2$')
plt.title('Samples from Correlated Gaussian via Cholesky')
plt.axis('equal')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

---
## Exercise 7: Determinant from Cholesky

Compute $\det(A)$ using Cholesky for:

$$A = \begin{pmatrix} 9 & 3 \\ 3 & 5 \end{pmatrix}$$

In [ ]:
A = np.array([[9, 3],
              [3, 5]], dtype=float)

# det(A) = det(L L^T) = det(L)^2 = (prod of diagonal of L)^2

# YOUR CODE HERE

In [ ]:
# SOLUTION
A = np.array([[9, 3],
              [3, 5]], dtype=float)

print(f"A = \n{A}")

# Direct
det_direct = det(A)
print(f"\nDirect: det(A) = 9×5 - 3×3 = {det_direct}")

# From Cholesky
L = cholesky(A)
print(f"\nCholesky L = \n{np.round(L, 4)}")
print(f"Diagonal of L: {np.round(np.diag(L), 4)}")

det_chol = np.prod(np.diag(L))**2
print(f"\ndet(A) = (∏ l_ii)² = ({np.diag(L)[0]:.4f} × {np.diag(L)[1]:.4f})²")
print(f"       = {np.prod(np.diag(L)):.4f}² = {det_chol:.4f}")

---
## Exercise 8: LDLᵀ Decomposition

Find the $LDL^T$ decomposition for:

$$A = \begin{pmatrix} 4 & 2 \\ 2 & 5 \end{pmatrix}$$

where $L$ is unit lower triangular and $D$ is diagonal.

In [ ]:
A = np.array([[4, 2],
              [2, 5]], dtype=float)

# d_1 = a_11
# l_21 = a_21 / d_1
# d_2 = a_22 - l_21^2 * d_1

# YOUR CODE HERE

In [ ]:
# SOLUTION
A = np.array([[4, 2],
              [2, 5]], dtype=float)

print(f"A = \n{A}")

print("\n--- LDLᵀ where L is unit lower triangular ---")

# d₁ = a₁₁
d1 = A[0, 0]
print(f"d₁ = a₁₁ = {d1}")

# l₂₁ = a₂₁/d₁
l21 = A[1, 0] / d1
print(f"l₂₁ = a₂₁/d₁ = 2/4 = {l21}")

# d₂ = a₂₂ - l₂₁²·d₁
d2 = A[1, 1] - l21**2 * d1
print(f"d₂ = a₂₂ - l₂₁²·d₁ = 5 - 0.25×4 = {d2}")

L = np.array([[1, 0],
              [l21, 1]])
D = np.diag([d1, d2])

print(f"\nL = \n{L}")
print(f"D = \n{D}")

print(f"\nVerification: L @ D @ Lᵀ = \n{L @ D @ L.T}")

print("\nRelation to Cholesky: A = (L√D)(L√D)ᵀ")
L_chol = L @ np.diag(np.sqrt([d1, d2]))
print(f"L_chol = \n{np.round(L_chol, 4)}")

---
## Exercise 9: Condition Number Impact

Compare solutions for $Ax = b$ with:
- $A_1 = I$ (well-conditioned)
- $A_2 = \begin{pmatrix} 1 & 1 \\ 1 & 1+10^{-10} \end{pmatrix}$ (ill-conditioned)
- $b = [1, 1]^T$

Observe the effect of small perturbation in $b$.

In [ ]:
A1 = np.array([[1, 0], [0, 1]], dtype=float)
A2 = np.array([[1, 1], [1, 1 + 1e-10]], dtype=float)
b = np.array([1, 1], dtype=float)

# Compare condition numbers and sensitivity to perturbation

# YOUR CODE HERE

In [ ]:
# SOLUTION
A1 = np.array([[1, 0], [0, 1]], dtype=float)
A2 = np.array([[1, 1], [1, 1 + 1e-10]], dtype=float)
b = np.array([1, 1], dtype=float)

print("--- Well-conditioned System (A = I) ---")
print(f"A₁ = \n{A1}")
print(f"κ(A₁) = {np.linalg.cond(A1):.1f}")
x1 = solve(A1, b)
print(f"Solution: x = {x1}")

print("\n--- Ill-conditioned System ---")
print(f"A₂ = \n{A2}")
print(f"κ(A₂) = {np.linalg.cond(A2):.2e}")
x2 = solve(A2, b)
print(f"Solution: x = {np.round(x2, 4)}")

print("\n--- Effect of Small Perturbation ---")
b_perturbed = b + np.array([1e-10, 0])

x1_pert = solve(A1, b_perturbed)
x2_pert = solve(A2, b_perturbed)

print(f"Well-conditioned: Δx = {np.linalg.norm(x1_pert - x1):.2e}")
print(f"Ill-conditioned:  Δx = {np.linalg.norm(x2_pert - x2):.2e}")
print("\nIll-conditioned systems amplify errors!")

---
## Exercise 10: Ridge Regression via Cholesky

Solve $(X^TX + \lambda I)w = X^Ty$ using Cholesky:

$$X = \begin{pmatrix} 1 & 1 \\ 1 & 2 \\ 1 & 3 \end{pmatrix}, \quad y = \begin{pmatrix} 1 \\ 2 \\ 2 \end{pmatrix}, \quad \lambda = 0.1$$

In [ ]:
X = np.array([[1, 1],
              [1, 2],
              [1, 3]], dtype=float)
y = np.array([1, 2, 2], dtype=float)
lam = 0.1

# Steps:
# 1. Compute A = X^T X + λI
# 2. Compute b = X^T y
# 3. Cholesky: A = L L^T
# 4. Solve L z = b (forward)
# 5. Solve L^T w = z (backward)

# YOUR CODE HERE

In [ ]:
# SOLUTION
X = np.array([[1, 1],
              [1, 2],
              [1, 3]], dtype=float)
y = np.array([1, 2, 2], dtype=float)
lam = 0.1

print(f"X = \n{X}")
print(f"y = {y}")
print(f"λ = {lam}")

# Normal equations with regularization
XTX = X.T @ X
XTy = X.T @ y

print(f"\nXᵀX = \n{XTX}")
print(f"Xᵀy = {XTy}")

A = XTX + lam * np.eye(2)
print(f"\nXᵀX + λI = \n{A}")

# Verify PD
eigs = eigvalsh(A)
print(f"Eigenvalues: {np.round(eigs, 4)} (all > 0 ✓)")

# Cholesky solve
L = cholesky(A)
print(f"\nCholesky L = \n{np.round(L, 4)}")

# Forward: Lz = Xᵀy
z = solve(L, XTy)
# Backward: Lᵀw = z
w = solve(L.T, z)

print(f"\nSolution w = {np.round(w, 4)}")
print(f"Predictions: Xw = {np.round(X @ w, 4)}")
print(f"True y:      {y}")

In [ ]:
# Visualize the fit
x_vals = X[:, 1]
x_line = np.linspace(0.5, 3.5, 100)
y_line = w[0] + w[1] * x_line

plt.figure(figsize=(8, 5))
plt.scatter(x_vals, y, s=100, c='red', zorder=5, label='Data points')
plt.plot(x_line, y_line, 'b-', linewidth=2, label=f'Ridge fit: y = {w[0]:.3f} + {w[1]:.3f}x')
plt.xlabel('x'); plt.ylabel('y')
plt.title(f'Ridge Regression via Cholesky (λ = {lam})')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
## Summary

### Key Decompositions

| Decomposition | Formula | When to Use |
|---------------|---------|-------------|
| LU | $A = LU$ or $PA = LU$ | General square systems |
| QR | $A = QR$ | Least squares, rectangular matrices |
| Cholesky | $A = LL^T$ | SPD matrices (fastest) |
| LDLᵀ | $A = LDL^T$ | SPD, avoids square roots |

### ML Applications

| Problem | Best Method |
|---------|-------------|
| Linear regression | QR |
| Ridge regression | Cholesky |
| Gaussian sampling | Cholesky |
| Determinant | LU or Cholesky |

### Stability Considerations

- High condition number → amplified errors
- Use pivoting for LU on general matrices
- Regularization improves conditioning